In [1]:
import sys
import os
import json
import numpy as np
import faiss
import pymupdf
import ollama 
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings('ignore')

OLLAMA_MODEL = "llama3.1"

embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print(f"Baza gotowa. Wybrany model generatora: {OLLAMA_MODEL}")

Baza gotowa. Wybrany model generatora: llama3.1


In [2]:
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []
print('Number of chunks: ', index.ntotal)

Number of chunks:  0


In [3]:
class Utils:
    def __init__(self, embedding_model, index, metadata, ollama_model, chunk_size=512):
        self.embedding_model = embedding_model
        self.index = index
        self.metadata = metadata
        self.ollama_model = ollama_model 
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            self.index.add(np.array([embeddings]))
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0
        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)
    
    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)
        D, I = self.index.search(np.array([question_embedding]), k)
        chunks = [self.metadata[i] for i in I[0]]

        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        prompt = prompt_template.format(context=context, query=query)

        messages = [
            {
                "role": "system",
                "content": "Be helpful, straight to the point. Use only context. Do not hallucinate."
            },
            {"role": "user", "content": prompt}
        ]

        response = ollama.chat(
            model=self.ollama_model,
            messages=messages,
            options={
                "temperature": temp,
                "num_predict": max_tokens
            }
        )

        answer = response['message']['content']
        return answer, chunks

In [4]:
utils = Utils(
    embedding_model=embedder,
    index=index,
    metadata=metadata,
    ollama_model=OLLAMA_MODEL, 
    chunk_size=512
)

In [5]:
knowledge_dir = "knowledge/"
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)
    
print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████████████████████████████████████████████████████| 70/70 [00:01<00:00, 56.56it/s]

Number of chunks:  3583


In [8]:
questions = [
    "Jakie są szanse procentowe na wylosowanie 1. numeru w drafcie przez czwartą najgorszą drużynę w lidze?",
    "Czym różni się błąd kroków w NBA w porównaniu do zasad FIBA?",
    "Ile wynosi maksymalny dozwolony czas trwania akcji w NBA zanim nastąpi błąd zegara?",
    "Jakie są warunki kwalifikujące gracza zagranicznego do przystąpienia do draftu NBA?",
    "Który rocznik Draft Combine z ostatnich lat wykazał najwyższą średnią rozpiętość ramion u rozgrywających?"
]

In [9]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Który rocznik Draft Combine z ostatnich lat wykazał najwyższą średnią rozpiętość ramion u rozgrywających?

**Odpowiedź:**

Relevant pasażaże:

* 2025-26: ACE BAILEY SF-PF -% 8.50 9.00 6' 7.50''
* 2024-25: REECE BEEKMAN PG -% 8.50 9.00 6' 1.25''
* 2023-24: ANTHONY BLACK PG -% 8.25 9.50 6' 5.75''

Odpowiedź:
 Reece Beekman z rocznika 2024-25 wykazał najwyższą średnią rozpiętość ramion u rozgrywających.

---
**Źródła:**

**[1]** `Draft Combine Anthrometric _ 26-27.pdf` — strona 1

> SEASON Players Draft Combine Anthro 2026-27 GLOSSARY 78 Rows • Page of 1 1 PLAYER POS BODY FAT % HAND LENGTH (INCHES) HAND WIDTH (INCHES) HEIGHT W/O SHOES HEIGHT W/ SH MATTHEW ABLE SG -% 8.75 10.50 6'...

**[2]** `Draft Combine Anthrometric _ 25-26.pdf` — strona 1

> SEASON Season: 2025-26 Players Draft Combine Anthro 2025-26 GLOSSARY 79 Rows • Page of 1 1 PLAYER POS BODY FAT % HAND LENGTH (INCHES) HAND WIDTH (INCHES) HEIGHT W/O SHOES HEIGHT W/  IZAN ALMANSA C -%...

**[3]** `Draft Combine Anthrometric _ 23-24.pdf` — strona 1

> 9.00 10.00 6' 7.00'' SIDY CISSOKO SF -% 9.25 10.00 6' 5.50'' JAYLEN CLARK SG -% 8.50 9.00 6' 4.00'' NOAH CLOWNEY PF -% - - BILAL COULIBALY SG-SF -% - - RICKY COUNCIL IV SF -% 7.75 8.50 6' 5.00'' GRAD...

**[4]** `Draft Combine Anthrometric _ 24-25.pdf` — strona 1

> SEASON Season: 2024-25 Players Draft Combine Anthro 2024-25 GLOSSARY 83 Rows • Page of 1 1 PLAYER POS BODY FAT % HAND LENGTH (INCHES) HAND WIDTH (INCHES) HEIGHT W/O SHOES HEIGHT W/  MICHAEL AJAYI SF -...

**[5]** `Draft Combine Anthrometric _ 23-24.pdf` — strona 1

> SEASON Season: 2023-24 Players Draft Combine Anthro 2023-24 GLOSSARY 81 Rows • Page of 1 1 PLAYER POS BODY FAT % HAND LENGTH (INCHES) HAND WIDTH (INCHES) HEIGHT W/O SHOES HEIGHT W/ S TREY ALEXANDER SG...